# (노트) 추천시스템 (IMDB) 

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- hide: false
- categories: [빅데이터분석]

### import 

In [1]:
import torch 
import matplotlib.pyplot as plt

In [2]:
from fastai.collab import * 
from fastai.tabular.all import * 

### data 

In [3]:
path = untar_data(URLs.ML_100k)

`-` 첫번째 자료 

In [7]:
ratings = pd.read_csv(path/'u.data', delimiter = '\t', header= None, names=['user','movie','rating','timestamp']) 
ratings.head()

,user,movie,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


- 마지막 열은 무의미 

`-` 두번째 자료 

In [6]:
movies = pd.read_csv(path/'u.item', delimiter = '|', encoding='latin-1', usecols=(0,1), names=('movie','title'), header=None)
movies.head()

,movie,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


`-` 두 데이터를 합친다. 

In [103]:
df = ratings.merge(movies)
df

,user,movie,rating,timestamp,title
0,196,242,3,881250949,Kolya (1996)
1,63,242,3,875747190,Kolya (1996)
2,226,242,5,883888671,Kolya (1996)
3,154,242,3,879138235,Kolya (1996)
4,306,242,5,876503793,Kolya (1996)
...,...,...,...,...,...
99995,840,1674,4,891211682,Mamma Roma (1962)
99996,655,1640,3,888474646,"Eighth Day, The (1996)"
99997,655,1637,3,888984255,Girls Town (1996)
99998,655,1630,3,887428735,"Silence of the Palace, The (Saimt el Qusur) (1994)"


### dls

In [104]:
dls = CollabDataLoaders.from_df(df,bs=64,item_name='title') 
dls.show_batch()

,user,title,rating
0,716,Brazil (1985),2
1,493,Mystery Science Theater 3000: The Movie (1996),4
2,758,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",5
3,7,For Whom the Bell Tolls (1943),5
4,299,"Awfully Big Adventure, An (1995)",1
5,391,Babe (1995),3
6,102,Judge Dredd (1995),2
7,116,William Shakespeare's Romeo and Juliet (1996),3
8,256,"Firm, The (1993)",3
9,746,Star Trek IV: The Voyage Home (1986),1


### learn 

In [105]:
lrnr = collab_learner(dls, n_factors=10, y_range=(0,5)) ## 책에서는 5.5 를 추천함 (실제로 5.5가 더 좋을것같으나 실험해보니 그렇게 큰차이는 없었음) 
lrnr.fit(13)

epoch,train_loss,valid_loss,time
0,1.144408,1.115417,00:03
1,0.888789,0.928337,00:04
2,0.902575,0.895762,00:03
3,0.839225,0.880892,00:03
4,0.833510,0.870261,00:03
5,0.805151,0.861027,00:03
6,0.822574,0.852714,00:03
7,0.811008,0.844709,00:03
8,0.786228,0.838482,00:03
9,0.785231,0.834099,00:03


- 책의 loss는 0.82근처 

In [106]:
lrnr.show_results()

,user,title,rating,rating_pred
0,350,760,5,4.343721
1,325,1366,1,3.184485
2,846,1300,5,2.879100
3,889,687,2,2.319529
4,643,144,5,3.252865
5,306,528,3,3.106971
6,62,559,2,1.596833
7,144,461,4,3.767398
8,417,1532,3,3.106261


- 솔직히 다맞추는 느낌은 없어요 

### learn2 

In [107]:
lrnr2 = collab_learner(dls, use_nn=True, y_range=(0,5),layers=[20,10]) ## 책에서는 5.5 를 추천함 (실제로 5.5가 더 좋을것같으나 실험해보니 그렇게 큰차이는 없었음) 
lrnr2.fit(8)

epoch,train_loss,valid_loss,time
0,0.947527,0.918384,00:03
1,0.880805,0.891152,00:04
2,0.851463,0.877467,00:03
3,0.822580,0.879698,00:04
4,0.781115,0.880758,00:04
5,0.757649,0.886081,00:04
6,0.721950,0.888590,00:04
7,0.725454,0.902457,00:04


In [108]:
lrnr2.show_results()

,user,title,rating,rating_pred
0,853,32,3,2.487452
1,526,1211,3,2.619166
2,671,1566,3,3.420372
3,864,260,2,3.322237
4,417,7,3,3.857708
5,933,742,4,3.618955
6,294,16,2,3.704982
7,655,1136,3,3.281573
8,245,694,5,3.250073


In [109]:
lrnr2.model

EmbeddingNN(
  (embeds): ModuleList(
    (0): Embedding(944, 74)
    (1): Embedding(1665, 102)
  )
  (emb_drop): Dropout(p=0.0, inplace=False)
  (bn_cont): BatchNorm1d(0, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layers): Sequential(
    (0): LinBnDrop(
      (0): Linear(in_features=176, out_features=20, bias=False)
      (1): ReLU(inplace=True)
      (2): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): LinBnDrop(
      (0): Linear(in_features=20, out_features=10, bias=False)
      (1): ReLU(inplace=True)
      (2): BatchNorm1d(10, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (2): LinBnDrop(
      (0): Linear(in_features=10, out_features=1, bias=True)
    )
    (3): SigmoidRange(low=0, high=5)
  )
)

### Bias 

In [113]:
lrnr.model

EmbeddingDotBias(
  (u_weight): Embedding(944, 10)
  (i_weight): Embedding(1665, 10)
  (u_bias): Embedding(944, 1)
  (i_bias): Embedding(1665, 1)
)

In [114]:
lrnr.model.i_bias.weight.detach().to('cpu').squeeze()

tensor([ 0.0009, -0.0355,  0.0572,  ...,  0.0850,  0.1988,  0.0905])

- 의미? 아이템의 바이어스, 클수록 평균적으로 좋은 평점을 가짐. $\to$ 취향을 떠나서 좋은 영화이다. 

In [141]:
lst1= lrnr.model.i_bias.weight.detach().to('cpu').squeeze().argsort(descending=True)[:20].tolist()
lst1 

[319,
 1501,
 1216,
 1581,
 1282,
 99,
 830,
 93,
 1318,
 1330,
 1653,
 622,
 1598,
 210,
 1528,
 1078,
 142,
 185,
 1399,
 1315]

- 평점이 높은 영화의 인덱스 

In [142]:
lst2= lrnr.model.i_bias.weight.detach().to('cpu').squeeze().argsort()[:20].tolist()
lst2

[1251,
 1574,
 295,
 62,
 850,
 1420,
 61,
 692,
 202,
 828,
 9,
 169,
 806,
 158,
 561,
 450,
 800,
 357,
 1452,
 775]

In [131]:
dls.classes.keys()

dict_keys(['user', 'title'])

In [143]:
list(dls.classes['title'][lst1])

['Close Shave, A (1995)',
 'Titanic (1997)',
 'Rear Window (1954)',
 'Vertigo (1958)',
 "Schindler's List (1993)",
 'As Good As It Gets (1997)',
 'L.A. Confidential (1997)',
 'Apt Pupil (1998)',
 'Shawshank Redemption, The (1994)',
 'Silence of the Lambs, The (1991)',
 'Wrong Trousers, The (1993)',
 'Good Will Hunting (1997)',
 'Wallace & Gromit: The Best of Aardman Animation (1996)',
 'Boot, Das (1981)',
 'Treasure of the Sierra Madre, The (1948)',
 'North by Northwest (1959)',
 'Beautiful Thing (1996)',
 'Blade Runner (1982)',
 'Star Wars (1977)',
 'Shall We Dance? (1996)']

In [144]:
list(dls.classes['title'][lst2])

['Robocop 3 (1993)',
 'Vampire in Brooklyn (1995)',
 'Children of the Corn: The Gathering (1996)',
 'Amityville 3-D (1983)',
 'Lawnmower Man 2: Beyond Cyberspace (1996)',
 'Striptease (1996)',
 "Amityville 1992: It's About Time (1992)",
 'Home Alone 3 (1997)',
 'Body Parts (1991)',
 'Kull the Conqueror (1997)',
 '3 Ninjas: High Noon At Mega Mountain (1998)',
 'Big Bully (1996)',
 'Kazaam (1996)',
 'Best of the Best 3: No Turning Back (1995)',
 'Free Willy 3: The Rescue (1997)',
 'Ed (1996)',
 'Jury Duty (1995)',
 'Crow: City of Angels, The (1996)',
 'Tales from the Hood (1995)',
 'Jaws 2 (1978)']

- 맞는것 같음 

### 예측 

타이타닉의 인덱스? 1501

In [67]:
x,y = dls.one_batch()

In [145]:
x

tensor([[ 125,  781],
        [ 293,   55],
        [ 825,  456],
        [ 556,  172],
        [ 339,   29],
        [ 303,  517],
        [ 479,  181],
        [ 189,  162],
        [ 707,  602],
        [ 308,  581],
        [ 545,  542],
        [ 796,  203],
        [ 625,  486],
        [ 279,  434],
        [ 588,  723],
        [ 285,  222],
        [ 234,  609],
        [ 696,  234],
        [ 648,  164],
        [  59,  918],
        [ 747,   93],
        [ 168,  259],
        [ 311,  645],
        [ 650,  419],
        [ 831,  328],
        [ 474,  847],
        [  97,  408],
        [ 326,    4],
        [ 796, 1100],
        [ 406,  769],
        [ 619,  273],
        [  42,  283],
        [ 326,  428],
        [ 896, 1264],
        [ 880, 1416],
        [ 634,  865],
        [ 514,  114],
        [ 374,  642],
        [  23,   90],
        [ 181, 1197],
        [ 608,  660],
        [ 214, 1072],
        [ 184,  428],
        [ 527,  467],
        [ 637, 1255],
        [ 

In [160]:
xx=torch.tensor([[i,1501] for i in range(1,31)])

- 1~30의 유저가 타이타닉에 대하여 어떻게 평가할지? 

In [164]:
lrnr.model(xx.to("cuda:0"))

tensor([4.2686, 4.4544, 3.0524, 4.7539, 3.6886, 3.5257, 4.5006, 4.6108, 4.7771,
        4.5230, 4.0832, 4.7357, 4.1658, 4.2745, 3.9919, 4.7964, 3.4304, 4.0366,
        3.9888, 3.7259, 4.1018, 4.5667, 3.9662, 4.4830, 4.4823, 3.8008, 4.0469,
        4.3801, 4.1530, 4.4030], device='cuda:0', grad_fn=<AddBackward0>)

In [165]:
lrnr2.model(xx.to("cuda:0")).reshape(-1)

tensor([4.4965, 4.5979, 3.8717, 4.8172, 4.1789, 4.3877, 4.4304, 4.7050, 4.5824,
        4.4999, 4.0471, 4.6527, 4.0895, 4.5800, 4.1738, 4.5995, 4.2151, 4.6460,
        4.0547, 3.7409, 4.4541, 4.6568, 4.4016, 4.5450, 4.4320, 4.1036, 4.0527,
        4.2717, 4.3840, 4.7297], device='cuda:0',
       grad_fn=<ReshapeAliasBackward0>)

`-` 로보캅3의 인덱스 

In [154]:
lst2[0]

1251

In [167]:
xx=torch.tensor([[i,1251] for i in range(1,31)])

In [168]:
lrnr.model(xx.to("cuda:0"))

tensor([1.0298, 1.2576, 2.0074, 1.8270, 1.5607, 1.2361, 1.8918, 1.3106, 1.6587,
        2.1011, 1.9014, 1.8065, 1.1476, 1.8122, 1.3259, 1.2895, 1.5796, 1.6705,
        1.8603, 2.1604, 0.8472, 0.7753, 1.5065, 1.5148, 2.3032, 1.3306, 1.6343,
        1.8864, 1.7526, 2.1888], device='cuda:0', grad_fn=<AddBackward0>)

- 1~30의 유저가 로보캅3에 대하여 어떻게 평가할지? 

In [169]:
lrnr2.model(xx.to("cuda:0")).reshape(-1)

tensor([0.9024, 1.4691, 1.3822, 3.3388, 1.3421, 1.3586, 1.6947, 1.4326, 2.3575,
        2.0974, 1.5968, 2.7778, 1.3750, 2.0561, 0.9425, 1.7809, 1.1335, 1.7297,
        1.9304, 1.1326, 0.8780, 1.0661, 1.1549, 1.2961, 1.9886, 1.3203, 1.2490,
        1.6458, 1.4746, 2.6102], device='cuda:0',
       grad_fn=<ReshapeAliasBackward0>)

In [173]:
dls.classes['title'][1461]

'Terminator 2: Judgment Day (1991)'

In [174]:
dls.classes['title'][1462]

'Terminator, The (1984)'